# Notebook 11: Training a Mini-GPT from Scratch

## Why This Matters

Reading about transformers is one thing; watching a transformer learn to predict text in real time is another. This notebook trains a character-level language model end-to-end — from raw text to generated output — implementing every component from scratch: tokenizer, architecture, training loop, sampling. The model you train here is architecturally identical to early GPT models, just smaller. When you watch the loss curve descend and see the model's output go from random characters to recognizable language patterns, you develop the physical intuition for what "the model is learning" that no amount of reading can provide. This is also the complete working example for "implement GPT from scratch" — a common technical interview task.

## Background

Character-level language models were popularized by Andrej Karpathy's "The Unreasonable Effectiveness of Recurrent Neural Networks" (2015). The model learns to predict the next character given all previous characters. Despite the simple objective, the model must learn:
- Spelling conventions
- Word boundaries
- Grammar and sentence structure
- Higher-level patterns (verse structure in poetry, argument structure in prose)

**Scaling laws (Kaplan et al. 2020):** Model performance (measured by cross-entropy loss) follows predictable power laws as a function of:
- $N$: number of parameters
- $D$: dataset tokens seen
- $C$: compute budget (FLOPs)

$$L(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N}, \quad L(D) \approx \left(\frac{D_c}{D}\right)^{\alpha_D}$$

The Chinchilla paper (Hoffmann et al. 2022) refined this: optimal training runs compute-budget-optimal with $D \approx 20N$ tokens.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time
import random
import os

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
random.seed(42)
print("Ready.")

## 1. Dataset and Tokenizer

We'll use a small public-domain text as our training corpus. Character-level tokenization is the simplest possible tokenizer: each unique character is a token. The vocabulary size is the number of unique characters (~65 for English text).

In [ ]:
# Use a small built-in corpus — Shakespeare sonnets (classic benchmark)
# This is the complete sonnets, ~100KB
text = """
From fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
Making a famine where abundance lies,
Thyself thy foe, to thy sweet self too cruel.
Thou that art now the world's fresh ornament
And only herald to the gaudy spring,
Within thine own bud buriest thy content
And, tender churl, mak'st waste in niggarding.
Pity the world, or else this glutton be,
To eat the world's due, by the grave and thee.

When forty winters shall beseige thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery, so gazed on now,
Will be a tatter'd weed, of small worth held:
Then being ask'd where all thy beauty lies,
Where all the treasure of thy lusty days,
To say, within thine own deep-sunken eyes,
Were an all-eating shame and thriftless praise.
How much more praise deserved thy beauty's use,
If thou couldst answer 'This fair child of mine
Shall sum my count and make my old excuse,'
Proving his beauty by succession thine.
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold.

Look in thy glass, and tell the face thou viewest
Now is the time that face should form another;
Whose fresh repair if now thou not renewest,
Thou dost beguile the world, unbless some mother.
For where is she so fair whose unear'd womb
Disdains the tillage of thy husbandry?
Or who is he so fond will be the tomb
Of his self-love, to stop posterity?
Thou art thy mother's glass, and she in thee
Calls back the lovely April of her prime;
So thou through windows of thine age shalt see,
Despite of wrinkles, this thy golden time.
But if thou live, remember'd not to be,
Die single, and thine image dies with thee.

Shall I compare thee to a summer's day?
Thou art more lovely and more temperate:
Rough winds do shake the darling buds of May,
And summer's lease hath all too short a date:
Sometime too hot the eye of heaven shines,
And often is his gold complexion dimm'd;
And every fair from fair sometime declines,
By chance, or nature's changing course, untrimm'd;
But thy eternal summer shall not fade,
Nor lose possession of that fair thou ow'st;
Nor shall Death brag thou wander'st in his shade,
When in eternal lines to time thou grow'st.
So long as men can breathe, or eyes can see,
So long lives this, and this gives life to thee.
""" * 20  # Repeat to get ~40KB of training data

# Build character-level vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}  # string → index
itos = {i: c for c, i in stoi.items()}      # index → string

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

# Train/val split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f"Text length: {len(text):,} characters")
print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")
print(f"Vocabulary: {''.join(chars[:30])}...")

## 2. Data Loading: Batched Random Sampling

For language model training, we sample random windows of length `block_size` from the training data. Each window provides `block_size` training examples (predict token at each position).

In [ ]:
block_size = 128    # Context window size
batch_size = 32    # Batch size

def get_batch(split, device):
    data = train_data if split == 'train' else val_data
    # Sample random starting positions
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# Verify
xb, yb = get_batch('train', device)
print(f"Input batch:  {xb.shape}  (batch_size={batch_size}, block_size={block_size})")
print(f"Target batch: {yb.shape}")
print(f"\nSample input  (first 20 chars): '{decode(xb[0, :20].tolist())}'")
print(f"Sample target (first 20 chars): '{decode(yb[0, :20].tolist())}'")
print("(Target is input shifted by 1 — next character prediction)")

## 3. Mini-GPT Architecture

A character-level GPT: token embedding + positional embedding + N decoder layers + LM head. We'll use a Pre-LN design with standard MHA (no GQA needed at this scale).

In [ ]:
class MiniGPTBlock(nn.Module):
    """Single GPT decoder layer: causal self-attention + FFN with Pre-LN."""
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout,
                                           batch_first=True)
        d_ff = 4 * d_model
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        # Precompute causal mask
        self.register_buffer('causal_mask',
            torch.triu(torch.ones(block_size, block_size), diagonal=1).bool())

    def forward(self, x):
        T = x.size(1)
        mask = self.causal_mask[:T, :T]
        residual = x
        x_norm = self.ln1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm,
                                 attn_mask=mask, is_causal=True)
        x = residual + attn_out

        residual = x
        x = residual + self.ffn(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding   = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            MiniGPTBlock(d_model, n_heads, block_size, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)
        print(f"MiniGPT: {sum(p.numel() for p in self.parameters()):,} parameters")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size

        tok_emb = self.token_embedding(idx)
        pos_emb = self.pos_embedding(torch.arange(T, device=idx.device))
        x = self.drop(tok_emb + pos_emb)

        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            return logits, None

        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# Create model
model = MiniGPT(
    vocab_size=vocab_size,
    d_model=128,
    n_heads=4,
    n_layers=4,
    block_size=block_size,
    dropout=0.1
).to(device)

## 4. Training Loop

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=50):
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        batch_losses = []
        for _ in range(eval_iters):
            x, y = get_batch(split, device)
            _, loss = model(x, y)
            batch_losses.append(loss.item())
        losses[split] = np.mean(batch_losses)
    model.train()
    return losses

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1,
                               betas=(0.9, 0.95))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2000)

# Training loop
max_iters = 2000
eval_interval = 200
train_losses = []
val_losses = []
eval_steps = []

print(f"Training for {max_iters} iterations...")
t0 = time.time()
for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(model)
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        eval_steps.append(step)
        elapsed = time.time() - t0
        print(f"step {step:4d}: train_loss={losses['train']:.4f}, "
              f"val_loss={losses['val']:.4f}, time={elapsed:.1f}s")

    x, y = get_batch('train', device)
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

print(f"\nTraining complete. Final val_loss: {val_losses[-1]:.4f}")
print(f"Initial val_loss: {val_losses[0]:.4f}")
print(f"Improvement: {val_losses[0] - val_losses[-1]:.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(eval_steps, train_losses, 'b-o', markersize=5, label='Train')
axes[0].plot(eval_steps, val_losses, 'r-o', markersize=5, label='Val')
axes[0].set_xlabel("Training step")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].set_title("Mini-GPT Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Perplexity
train_ppl = [math.exp(l) for l in train_losses]
val_ppl   = [math.exp(l) for l in val_losses]
axes[1].plot(eval_steps, train_ppl, 'b-o', markersize=5, label='Train')
axes[1].plot(eval_steps, val_ppl, 'r-o', markersize=5, label='Val')
axes[1].set_xlabel("Training step")
axes[1].set_ylabel("Perplexity")
axes[1].set_title("Perplexity (lower = better)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Mini-GPT on Shakespeare", fontsize=12)
plt.tight_layout()
plt.savefig("mini_gpt_training.png", dpi=120, bbox_inches='tight')
plt.show()

## 5. Text Generation with Temperature Sampling

After training, we can generate text by sampling from the model's distribution. **Temperature** controls the sharpness of the distribution:

$$P_T(x_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- $T = 0$: greedy (always pick the most likely token)
- $T = 1$: sample from the model's true learned distribution
- $T > 1$: more diverse/random — flattens the distribution
- $T < 1$: more focused/conservative — sharpens the distribution

**Top-k sampling:** Only sample from the top-$k$ most probable tokens, setting all others to 0 probability. Prevents catastrophically low-probability tokens from being selected.

**Nucleus (top-p) sampling:** Sample from the smallest set of tokens whose cumulative probability exceeds $p$. Adapts the number of candidate tokens based on the local distribution shape — uses fewer candidates when the model is confident, more when it's uncertain.

In [ ]:
@torch.no_grad()
def generate(model, prompt_text, max_new_tokens=200, temperature=1.0, top_k=None):
    """Generate text from a prompt using the trained model."""
    model.eval()
    context = torch.tensor([encode(prompt_text)], dtype=torch.long, device=device)

    generated_chars = []
    for _ in range(max_new_tokens):
        # Crop to block_size
        context_cond = context[:, -model.block_size:]
        logits, _ = model(context_cond)
        logits = logits[:, -1, :] / temperature  # (1, vocab_size)

        if top_k is not None:
            # Zero out all but top-k logits
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        context = torch.cat([context, next_token], dim=1)
        generated_chars.append(itos[next_token.item()])

    return prompt_text + ''.join(generated_chars)

# Generate at different temperatures
print("="*60)
for temp in [0.5, 1.0, 1.5]:
    print(f"\n--- Temperature {temp} ---")
    generated = generate(model, "Shall I ", max_new_tokens=150,
                          temperature=temp, top_k=20)
    print(generated)

## 6. Visualizing Learned Attention Patterns

After training, we can inspect what the model has learned. Different heads should specialize in different patterns.

In [ ]:
# Extract attention weights from the trained model
def get_attention_weights(model, text, layer=0, device=device):
    model.eval()
    tokens = torch.tensor([encode(text[:block_size])], dtype=torch.long,
                           device=device)
    B, T = tokens.shape

    # Hook into the attention layer
    attn_weights_storage = []

    def hook_fn(module, input, output):
        # output[1] is the attention weights from nn.MultiheadAttention
        if output[1] is not None:
            attn_weights_storage.append(output[1].detach().cpu())

    handle = model.blocks[layer].attn.register_forward_hook(hook_fn)

    with torch.no_grad():
        model(tokens)

    handle.remove()
    return attn_weights_storage[0][0], decode(tokens[0, :T].tolist())

sample_text = "Shall I compare thee to a summer"
attn_w, chars = get_attention_weights(model, sample_text, layer=0)
T = len(chars)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(attn_w[:T, :T].numpy(), cmap='Blues', aspect='auto')
ax.set_xticks(range(T))
ax.set_yticks(range(T))
ax.set_xticklabels(list(chars), fontsize=9)
ax.set_yticklabels(list(chars), fontsize=9)
ax.set_title(f"Attention Pattern (averaged heads, Layer 0)
'{sample_text}'")
ax.set_xlabel("Key (attended to)")
ax.set_ylabel("Query")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig("mini_gpt_attention.png", dpi=120, bbox_inches='tight')
plt.show()

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Character-level LM | Each character = one token; vocab ~65; simplest possible tokenizer |
| Next-token prediction | Cross-entropy loss at every position; teacher forcing during training |
| Scaling laws | $L \propto N^{-\alpha}$; loss follows power law vs params/data/compute |
| Chinchilla optimal | ~20 tokens per parameter for compute-optimal training |
| Temperature sampling | $T < 1$: focused; $T = 1$: natural; $T > 1$: random |
| Top-k sampling | Restrict to $k$ highest-prob tokens; prevents catastrophic samples |
| Top-p (nucleus) | Restrict to tokens summing to probability $p$; adapts to local confidence |
| Weight tying | LM head = token embedding transpose; saves params; improves perplexity |
| Context window | Hard limit: can only see `block_size` previous tokens during generation |
| Perplexity | $e^{\text{loss}}$; a model with perplexity 5 is "as uncertain as a 5-sided die per step" |